In [6]:
import botocore
import boto3
import pandas as pd
import duckdb
import os

# set variables
s3_region = os.getenv('S3_REGION')
s3_access_key_id = os.getenv('S3_ACCESS_KEY_ID')
s3_secret_access_key = os.getenv('S3_SECRET_ACCESS_KEY')

picks_season_path          = "s3://greglenane-drive-to-survive/picks/picks_season.parquet"
combined_scoring_path      = "s3://greglenane-drive-to-survive/results_scored.parquet"
picks_scored_path          = "s3://greglenane-drive-to-survive/picks_scored.parquet"
scoring_aggregate_path     = "s3://greglenane-drive-to-survive/scored_aggregate.parquet"
teams_path                 = "s3://greglenane-drive-to-survive/mapping/teams.csv"

In [7]:
# Enable S3 access
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
SET s3_region= '{s3_region}';
SET s3_access_key_id= '{s3_access_key_id}';
SET s3_secret_access_key= '{s3_secret_access_key}';
""")

In [8]:
picks = con.execute(f"""
                                
        SELECT *
        FROM '{picks_season_path}' 

    """).df()

In [9]:
print(picks)

        Name  Round       Date           Race             Driver source_sheet
0      Healy      4 2026-05-02       Miami GP        Liam Lawson            4
1   Lawrence      4 2026-05-02       Miami GP       Esteban Ocon            4
2      Vinny      4 2026-05-02       Miami GP    Nico Hülkenberg            4
3        Joe      4 2026-05-02       Miami GP     Oliver Bearman            4
4       Jake      4 2026-05-02       Miami GP       Isack Hadjar            4
5       Sean      4 2026-05-02       Miami GP     Arvid Lindblad            4
6       Greg      4 2026-05-02       Miami GP       Pierre Gasly            4
7      Henry      4 2026-05-02       Miami GP     Max Verstappen            4
8       Liam      4 2026-05-02       Miami GP   Franco Colapinto            4
9       Kyle      4 2026-05-02       Miami GP    Fernando Alonso            4
10       Sam      4 2026-05-02       Miami GP  Gabriel Bortoleto            4
11     Griff      4 2026-05-02       Miami GP    Valtteri Bottas

In [10]:
scored = con.execute(f"""
                                
        SELECT *
        FROM '{picks_scored_path}' 

    """).df()

In [11]:
print(scored)

        Name  Round       Date           Race             Driver source_sheet  \
0      Healy      2 2026-03-15       China GP     Oliver Bearman            2   
1       Liam      2 2026-03-15       China GP       Pierre Gasly            2   
2      Griff      2 2026-03-15       China GP        Liam Lawson            2   
3   Lawrence      2 2026-03-15       China GP       Isack Hadjar            2   
4        Joe      2 2026-03-15       China GP       Carlos Sainz            2   
5      Vinny      2 2026-03-15       China GP   Franco Colapinto            2   
6       Kyle      2 2026-03-15       China GP    Nico Hülkenberg            2   
7      Henry      2 2026-03-15       China GP     Arvid Lindblad            2   
8       Jake      2 2026-03-15       China GP    Valtteri Bottas            2   
9       Sean      2 2026-03-15       China GP       Esteban Ocon            2   
10       Sam      2 2026-03-15       China GP  Gabriel Bortoleto            2   
11      Greg      2 2026-03-

In [14]:
agg = con.execute(f"""
                                
        SELECT *
        FROM '{scoring_aggregate_path}' 
        where Name = 'Griff'

    """).df()

In [15]:
print(agg)

    Name  team    team_name  Round           Race       Date           Driver  \
0  Griff     3  Healy-Griff      1  Australian GP 2026-03-07     Isack Hadjar   
1  Griff     3  Healy-Griff      2       China GP 2026-03-15      Liam Lawson   
2  Griff     3  Healy-Griff      3       Japan GP 2026-03-28     Esteban Ocon   
3  Griff     3  Healy-Griff      4       Miami GP 2026-05-02  Valtteri Bottas   

        Constructor gp_grid  gp_expected  ... sprint_expected  \
0          Red Bull       3           -1  ...            <NA>   
1        RB F1 Team      14            2  ...               1   
2      Haas F1 Team      12            6  ...            <NA>   
3  Cadillac F1 Team      19            0  ...               0   

   sprint_position  sprint sprint_var  total total_expected  total_var  \
0             None    <NA>          0      0             -1          1   
1                7       1          0      5              3          2   
2             None    <NA>          0     10  